# Week 6 — RAG Evaluation

We've been eyeballing results until now. This notebook makes quality quantitative.

**Two types of evaluation:**

| Type | Metrics | What it measures |
|---|---|---|
| Retrieval | MRR, NDCG | Is the right ticket being found and ranked well? |
| Generation | RAGAS (Faithfulness, AnswerRelevancy) | Is the LLM answer grounded and relevant? |

**Eval dataset:** 100 resolved tickets sampled from the test set.
Ground truth relevance = same `sub_category` — a VPN query should retrieve VPN tickets.

## 1. Setup

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
from tqdm import tqdm
from src.retriever import retrieve

print("Imports OK")

## 2. Build the evaluation dataset

We need (query, ground_truth_subcategory) pairs.
We sample 100 resolved tickets — their descriptions are the queries,
their sub_category is the ground truth label for relevance.

In [ ]:
df = pd.read_csv("data/processed/tickets_clean.csv")
resolved = df[df["status"].isin(["Closed", "Resolved"])].copy()

# Sample 100 tickets — use a fixed seed for reproducibility
eval_df = resolved.sample(100, random_state=99).reset_index(drop=True)

print(f"Eval set: {len(eval_df)} tickets")
print(f"Sub-categories represented: {eval_df['sub_category'].nunique()}")
print()
print(eval_df[["category", "sub_category", "priority"]].value_counts().head(10))

## 3. Run retrieval on the eval set

For each query, we retrieve the top 10 candidates and record:
- the sub_category of each retrieved ticket
- whether each result is "relevant" (same sub_category as the query)

In [ ]:
TOP_K = 10  # retrieve top 10 for evaluation

results = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Retrieving"):
    query = row["description"]
    true_subcat = row["sub_category"]

    candidates = retrieve(query, top_k=TOP_K)

    # For each candidate, mark it relevant if it has the same sub_category.
    # sub_category is stored in ChromaDB metadata.
    relevance = []
    for c in candidates:
        # ChromaDB metadata uses the key name as stored in embeddings.py
        cand_subcat = c.get("sub_category", "") or c.get("category", "")
        relevance.append(1 if cand_subcat == true_subcat else 0)

    results.append({
        "query":         query[:80],
        "true_subcat":   true_subcat,
        "relevance":     relevance,   # list of 0/1, length = TOP_K
        "candidates":    candidates,
    })

print(f"Retrieval complete for {len(results)} queries.")

## 4. MRR — Mean Reciprocal Rank

For each query, find the rank of the first relevant result.
MRR is the mean of 1/rank across all queries.

- MRR = 1.0 → every query's first result is relevant (perfect)
- MRR = 0.5 → relevant result is at rank 2 on average
- MRR = 0.0 → no relevant results found in top-k

MRR answers: *"How quickly does the user find a relevant ticket?"*

In [ ]:
def mean_reciprocal_rank(results):
    rr_scores = []
    for r in results:
        rr = 0.0
        for rank, rel in enumerate(r["relevance"], start=1):
            if rel == 1:
                rr = 1.0 / rank  # reciprocal of the first relevant rank
                break
        rr_scores.append(rr)
    return np.mean(rr_scores), rr_scores

mrr, rr_scores = mean_reciprocal_rank(results)

# How many queries had a relevant result at each rank?
hits_at_k = {}
for k in [1, 3, 5, 10]:
    hits = sum(1 for r in results if any(r["relevance"][:k]))
    hits_at_k[k] = hits / len(results)

print(f"MRR@{TOP_K}: {mrr:.4f}")
print()
print("Hit rate (at least 1 relevant result in top-k):")
for k, rate in hits_at_k.items():
    print(f"  Hit@{k:2d}: {rate:.1%}")

## 5. NDCG — Normalized Discounted Cumulative Gain

NDCG rewards having **multiple** relevant results ranked highly.
It discounts results at lower ranks (rank 2 is worth half of rank 1).

- NDCG = 1.0 → all relevant results appear at the top (perfect ordering)
- NDCG = 0.0 → no relevant results found

NDCG answers: *"How well is the entire ranked list ordered by relevance?"*

In [ ]:
def dcg(relevances):
    """Discounted Cumulative Gain for a ranked list of relevance scores."""
    return sum(rel / np.log2(rank + 1) for rank, rel in enumerate(relevances, start=1))

def ndcg(results, k=10):
    scores = []
    for r in results:
        rel = r["relevance"][:k]
        actual_dcg = dcg(rel)
        # Ideal DCG = all relevant items at the top
        ideal_dcg = dcg(sorted(rel, reverse=True))
        scores.append(actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0)
    return np.mean(scores)

ndcg_5  = ndcg(results, k=5)
ndcg_10 = ndcg(results, k=10)

print(f"NDCG@5:  {ndcg_5:.4f}")
print(f"NDCG@10: {ndcg_10:.4f}")

## 6. Retrieval results summary

In [ ]:
import matplotlib.pyplot as plt

print("=" * 40)
print("  Retrieval Evaluation Summary")
print("=" * 40)
print(f"  MRR@{TOP_K}:   {mrr:.4f}")
print(f"  NDCG@5:  {ndcg_5:.4f}")
print(f"  NDCG@10: {ndcg_10:.4f}")
print(f"  Hit@1:   {hits_at_k[1]:.1%}")
print(f"  Hit@5:   {hits_at_k[5]:.1%}")
print(f"  Hit@10:  {hits_at_k[10]:.1%}")
print()

# Distribution of reciprocal ranks
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(rr_scores, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(mrr, color='red', linestyle='--', label=f'MRR = {mrr:.3f}')
axes[0].set_title('Reciprocal Rank Distribution')
axes[0].set_xlabel('Reciprocal Rank (1/rank of first relevant result)')
axes[0].legend()

# Hit rate by k
ks = list(hits_at_k.keys())
rates = list(hits_at_k.values())
axes[1].bar(ks, rates, color='coral', width=0.6)
axes[1].set_title('Hit Rate @ k')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Fraction of queries with ≥1 relevant result')
axes[1].set_ylim(0, 1)
for k, rate in zip(ks, rates):
    axes[1].text(k, rate + 0.02, f'{rate:.0%}', ha='center')

plt.tight_layout()
plt.show()

## 7. RAGAS — Generation Quality

RAGAS uses an LLM to judge the quality of RAG outputs.
We run it on 10 samples (slow with local Mistral — ~2 min total).

**Metrics:**
- **Faithfulness:** Does the answer only contain facts from the retrieved context? Scores hallucination.
- **Answer Relevancy:** Is the answer actually relevant to the question?

Both use Mistral via Ollama as the judge LLM — no OpenAI key needed.

In [ ]:
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from src.rag_chain import run

# Use Mistral via Ollama as both the judge LLM and embedding model
evaluator_llm = LangchainLLMWrapper(ChatOllama(model="mistral"))
evaluator_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="mistral"))

print("RAGAS evaluator ready")

In [ ]:
# Build RAGAS eval samples — 10 tickets is enough to demonstrate the framework
ragas_sample_df = eval_df.sample(10, random_state=7).reset_index(drop=True)

ragas_samples = []

for _, row in ragas_sample_df.iterrows():
    query = row["description"]
    ground_truth = row["resolution"]  # the actual resolution from the dataset

    # Run the full RAG chain
    result = run(query, provider="ollama", model_name="mistral")

    # RAGAS needs: user_input, response, retrieved_contexts, reference
    context_texts = [s["resolution"] for s in result["sources"] if s.get("resolution")]

    sample = SingleTurnSample(
        user_input=query,
        response=result["response"],
        retrieved_contexts=context_texts,
        reference=ground_truth,
    )
    ragas_samples.append(sample)
    print(f"  Collected sample {len(ragas_samples)}/10")

dataset = EvaluationDataset(samples=ragas_samples)
print("\nDataset ready. Running RAGAS evaluation...")

In [ ]:
from ragas import RunConfig

# Ollama/Mistral is slow — increase timeout to 180s per job to avoid TimeoutError
run_config = RunConfig(timeout=180, max_retries=3, max_wait=60)

ragas_results = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(), AnswerRelevancy()],
    llm=evaluator_llm,
    embeddings=evaluator_emb,
    run_config=run_config,
)

print(ragas_results)

In [ ]:
ragas_df = ragas_results.to_pandas()

print("Per-sample RAGAS scores:")
print(ragas_df[["faithfulness", "answer_relevancy"]].to_string())
print()
print("Averages:")
print(f"  Faithfulness:     {ragas_df['faithfulness'].mean():.4f}")
print(f"  Answer Relevancy: {ragas_df['answer_relevancy'].mean():.4f}")

## 8. Full Evaluation Summary

All metrics in one place.

In [ ]:
print("=" * 50)
print("  RAG EVALUATION REPORT")
print("=" * 50)
print()
print("RETRIEVAL (n=100 queries)")
print(f"  MRR@10:          {mrr:.4f}")
print(f"  NDCG@5:          {ndcg_5:.4f}")
print(f"  NDCG@10:         {ndcg_10:.4f}")
print(f"  Hit@1:           {hits_at_k[1]:.1%}")
print(f"  Hit@5:           {hits_at_k[5]:.1%}")
print()
print("GENERATION (n=10 samples, judge: Mistral via Ollama)")
print(f"  Faithfulness:    {ragas_df['faithfulness'].mean():.4f}")
print(f"  Answer Relevancy:{ragas_df['answer_relevancy'].mean():.4f}")
print()
print("EMBEDDING MODEL: BAAI/bge-small-en-v1.5")
print("RERANKER:        cross-encoder/ms-marco-MiniLM-L-6-v2")
print("LLM:             Mistral 7B via Ollama (local)")

## Notes & Limitations

- **Perfect retrieval scores:** MRR@10 = 1.0 and Hit@1 = 100% because the eval queries are sampled from the same pool as the indexed tickets — the retriever finds the exact source ticket as the top result. This validates that embeddings are consistent, but is not a true generalisation benchmark. A proper eval would use a held-out set of tickets *not* present in ChromaDB.
- **Relevance definition:** We used `sub_category` match as ground truth. This is a proxy — in production you'd have human-labelled relevance judgements.
- **RAGAS sample size:** 10 samples is enough to demonstrate the framework but too small for statistical significance. A real evaluation would use 200+ samples.
- **Faithfulness NaN:** Mistral's output format wasn't parseable by RAGAS for the faithfulness metric — a known issue with smaller local models. In production, use a stronger judge (GPT-4, Claude).
- **Answer Relevancy 0.51:** Mistral 7B sometimes produces verbose or tangential responses. A larger model or tighter prompt would improve this.
- **Embedding model comparison:** BGE-small vs BGE-large or OpenAI `text-embedding-3-small` would be the next experiment — BGE-large typically improves MRR by 5–10% at the cost of slower encoding.

**Week 7:** README, demo video, and deployment to Hugging Face Spaces or AWS EC2.